# **Лабораторная работа №2. Вывод модели в эксплуатацию**

**Вариант 5 -** Отказ оборудования

**Цель работы —** превратить модель, полученную в лабораторной работе № 1, в воспроизводимый сервис: с журналом экспериментов, REST API, автоматическими тестами, измеренной производительностью и контролем дрейфа данных.

## **Шаг 1 — Подготовка проекта**

### **Подключение Google Диска**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **Структура проекта**

In [2]:
import os, pathlib

PROJECT = '/content/drive/MyDrive/lab2_variant5'

for sub in ['src', 'tests', 'data', 'models', 'mlruns', 'reports']:
    pathlib.Path(f'{PROJECT}/{sub}').mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT)
print('Текущая директория:', os.getcwd())
print()
print('Структура:')
!ls -la

Текущая директория: /content/drive/MyDrive/lab2_variant5

Структура:
total 46
drwx------ 8 root root  4096 Oct 21 12:45 .
drwx------ 8 root root  4096 Oct 21 11:23 ..
drwx------ 2 root root  4096 Oct 21 12:45 data
drwx------ 4 root root  4096 Oct 21 12:41 mlruns
drwx------ 2 root root  4096 Oct 21 12:45 models
drwx------ 2 root root  4096 Oct 21 11:24 reports
-rw------- 1 root root 13528 Oct 21 12:45 requirements.txt
drwx------ 3 root root  4096 Oct 21 12:45 src
drwx------ 2 root root  4096 Oct 21 11:24 tests


### **Установка библиотек**

In [3]:
%pip uninstall -y gradio gradio-client
!pip cache purge

%pip install --no-cache-dir -q "gradio==5.40.0"

%pip install -q "fastapi==0.115.*" "uvicorn==0.32.*" "pydantic==2.*" \
    "mlflow==2.*" "evidently==0.4.*" "locust==2.*" \
    "scikit-learn==1.5.*" "lightgbm==4.*" "catboost==1.2.*" \
    nest-asyncio pytest httpx ucimlrepo

Found existing installation: gradio 5.40.0
Uninstalling gradio-5.40.0:
  Successfully uninstalled gradio-5.40.0
Found existing installation: gradio_client 1.11.0
Uninstalling gradio_client-1.11.0:
  Successfully uninstalled gradio_client-1.11.0
Files removed: 0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.5/324.5 kB 89.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.11.0 which is incompatible.


### **Проверка версий**

In [4]:
from gevent import monkey
monkey.patch_all()

True

In [3]:
import importlib.metadata as md

for pkg in ['fastapi', 'pydantic', 'mlflow', 'evidently', 'gradio',
            'scikit-learn', 'lightgbm', 'catboost', 'locust']:
    try:
        print(f'{pkg:12}: {md.version(pkg)}')
    except Exception as e:
        print(f'{pkg:12}: {type(e).__name__}: {e}')

fastapi     : 0.115.14
pydantic    : 2.11.10
mlflow      : 2.22.5
evidently   : 0.4.40
gradio      : 5.40.0
scikit-learn: 1.5.2
lightgbm    : 4.6.0
catboost    : 1.2.10
locust      : 2.46.7


In [ ]:
import os
PROJECT = '/content/drive/MyDrive/lab2_variant5'
os.chdir(PROJECT)

import importlib.metadata as md
print('gradio:', md.version('gradio'))
print('gradio-client:', md.version('gradio-client'))

import gradio as gr
print('gradio импортируется OK')
print('Версия gradio:', gr.__version__)

import fastapi, pydantic, mlflow, evidently, sklearn, lightgbm, catboost, locust
print(f'fastapi:   {fastapi.__version__}')
print(f'pydantic:  {pydantic.__version__}')
print(f'mlflow:    {mlflow.__version__}')
print(f'evidently: {evidently.__version__}')
print(f'sklearn:   {sklearn.__version__}')
print(f'lightgbm:  {lightgbm.__version__}')
print(f'catboost:  {catboost.__version__}')
print(f'locust:    {locust.__version__}')

gradio: 5.40.0
gradio-client: 1.11.0
gradio импортируется OK
Версия gradio: 5.40.0


/usr/local/lib/python3.13/dist-packages/locust/__init__.py:16: MonkeyPatchWarning: Monkey-patching ssl after ssl has already been imported may lead to errors, including RecursionError on Python 3.6. It may also silently lead to incorrect behaviour on Python 3.7. Please monkey-patch earlier. See https://github.com/gevent/gevent/issues/1016. Modules that had direct imports (NOT patched): ['urllib3.util.ssl_ (/usr/local/lib/python3.13/dist-packages/urllib3/util/ssl_.py)', 'urllib3.util (/usr/local/lib/python3.13/dist-packages/urllib3/util/__init__.py)']. 
  monkey.patch_all()


### **Фиксация окружения**

In [7]:
import subprocess

result = subprocess.run(['pip', 'freeze'], capture_output=True, text=True)

with open('requirements.txt', 'w') as f:
    f.write(result.stdout)

for line in result.stdout.splitlines():
    if any(line.startswith(p) for p in (
        'fastapi', 'uvicorn', 'pydantic', 'mlflow', 'evidently',
        'locust', 'gradio', 'scikit-learn', 'lightgbm', 'catboost',
    )):
        print(line)

Exception ignored in: <bound method _ForkHooks.after_fork_in_child of <gevent.threading._ForkHooks object at 0x7ad2af2bf770>>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 447, in after_fork_in_child
    self._stop_running_greenlets_in_child()
  File "/usr/local/lib/python3.13/dist-packages/gevent/threading.py", line 435, in _stop_running_greenlets_in_child
    handle._set_done(enter_hub=False)
TypeError: _ThreadHandle._set_done() takes no keyword arguments


catboost==1.2.10
evidently==0.4.40
fastapi==0.115.14
gradio==5.40.0
gradio_client==1.11.0
lightgbm==4.6.0
locust==2.46.7
mlflow==2.22.5
mlflow-skinny==2.22.5
pydantic==2.11.10
pydantic_core==2.33.2
scikit-learn==1.5.2
uvicorn==0.32.1


### **Загрузка данных варианта**

In [8]:
#Вариант 5 — AI4I 2020 Predictive Maintenance (UCI id=601)
from ucimlrepo import fetch_ucirepo
import pandas as pd
import re

ds = fetch_ucirepo(id=601)
df_raw = pd.concat([ds.data.features, ds.data.targets], axis=1)

def _norm(c):
    c = c.replace("'", "").replace("’", "")
    return re.sub(r'[^0-9a-zA-Z]+', '_', c).strip('_').lower()

df_raw.columns = [_norm(c) for c in df_raw.columns]
print('Колонки:', list(df_raw.columns))
print('Размер:', df_raw.shape)

#Сохраняем сырые данные на Диск
df_raw.to_csv('data/raw.csv', index=False)
print('Сохранено: data/raw.csv')

Колонки: ['type', 'air_temperature', 'process_temperature', 'rotational_speed', 'torque', 'tool_wear', 'machine_failure', 'twf', 'hdf', 'pwf', 'osf', 'rnf']
Размер: (10000, 12)
Сохранено: data/raw.csv


## **Шаг 2 — Модули проекта**

### **Создание src/features.py**

In [9]:
%%writefile src/features.py
"""Признаки варианта 5 (отказ оборудования).

Импортируется и ноутбуком (Шаг 3), и сервисом (Шаг 4),
и модулем предсказания (predict.py).
"""
import numpy as np
import pandas as pd

RANDOM_STATE = 42

#Категориальная колонка (единственная в наборе AI4I)
CAT_COLS = ['type']

#Сырые признаки (6 штук) — вход модели
RAW_FEATURES = [
    'type',
    'air_temperature',
    'process_temperature',
    'rotational_speed',
    'torque',
    'tool_wear',
]

#елевая переменная
TARGET = 'machine_failure'


def add_features(X: pd.DataFrame) -> pd.DataFrame:
    """Добавляет физические признаки.

    Из 5 кандидатов из ЛР1 оставлены 4 (power_per_wear удалён
    по результатам абляции). Именно на этих 4 обучена финальная
    модель LightGBM из ЛР1 — сохранение их гарантирует
    совместимость с model.joblib.
    """
    X = X.copy()

    #1. Мощность, Вт = момент (Н·м) × угловая скорость (рад/с)
    X['power_w'] = X['torque'] * X['rotational_speed'] * 2 * np.pi / 60

    #2. Разность температур: процесса и воздуха, К
    X['temp_diff_k'] = X['process_temperature'] - X['air_temperature']

    #3. Произведение износа на момент
    X['wear_x_torque'] = X['tool_wear'] * X['torque']

    #4. Отношение скорости к моменту (устойчиво к нулю)
    X['speed_per_torque'] = X['rotational_speed'] / (X['torque'] + 1.0)

    return X


def to_category(X: pd.DataFrame) -> pd.DataFrame:
    """Приводит type к category для LightGBM."""
    X = X.copy()
    for c in CAT_COLS:
        if c in X.columns:
            X[c] = X[c].astype('category')
    return X


def full_transform(X: pd.DataFrame) -> pd.DataFrame:
    """Полный конвейер: add_features + to_category."""
    return to_category(add_features(X))

Overwriting src/features.py


### **Создание src/train.py**

In [10]:
%%writefile src/train.py
"""Обучение модели LightGBM с препроцессингом."""
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_score, recall_score, f1_score,
)

from src.features import full_transform, RAW_FEATURES, TARGET, RANDOM_STATE


def train_model(X: pd.DataFrame, y: pd.Series, params: dict) -> Pipeline:
    """Обучает LightGBM за препроцессингом full_transform.

    Parameters
    ----------
    X : DataFrame с RAW_FEATURES (6 колонок)
    y : бинарная целевая (0/1)
    params : параметры LGBMClassifier

    Returns
    -------
    sklearn Pipeline (fe → model)
    """
    params = {**params, 'random_state': RANDOM_STATE, 'verbose': -1, 'n_jobs': -1}
    model = lgb.LGBMClassifier(**params)
    pipe = Pipeline([
        ('fe', FunctionTransformer(full_transform, validate=False)),
        ('model', model),
    ])
    pipe.fit(X, y)
    return pipe


def evaluate(pipe: Pipeline, X: pd.DataFrame, y: pd.Series) -> dict:
    """Считает метрики на выборке (PR-AUC, ROC-AUC, F1, P, R при пороге 0.5)."""
    proba = pipe.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        'pr_auc':  float(average_precision_score(y, proba)),
        'roc_auc': float(roc_auc_score(y, proba)),
        'f1':      float(f1_score(y, pred, zero_division=0)),
        'precision': float(precision_score(y, pred, zero_division=0)),
        'recall':  float(recall_score(y, pred, zero_division=0)),
    }

Overwriting src/train.py


### **Создание src/predict.py**

In [11]:
%%writefile src/predict.py
"""Инференс: загрузка сохранённой модели и предсказание по списку словарей."""
import json
import os
from typing import List

import joblib
import numpy as np
import pandas as pd

from src.features import RAW_FEATURES

DEFAULT_MODEL_PATH = os.environ.get('MODEL_PATH', 'models/model.joblib')
DEFAULT_SCHEMA_PATH = os.environ.get('SCHEMA_PATH', 'models/input_schema.json')
DEFAULT_THRESHOLD = 0.022

_MODEL = None
_SCHEMA = None
_THRESHOLD = None


def load_model(model_path: str = DEFAULT_MODEL_PATH,
               schema_path: str = DEFAULT_SCHEMA_PATH,
               threshold: float = DEFAULT_THRESHOLD) -> None:
    """Загружает модель один раз (вызывается при старте сервиса)."""
    global _MODEL, _SCHEMA, _THRESHOLD
    _MODEL = joblib.load(model_path)
    with open(schema_path, encoding='utf-8') as f:
        _SCHEMA = json.load(f)
    _THRESHOLD = float(threshold)


def get_threshold() -> float:
    return _THRESHOLD


def get_schema() -> dict:
    return _SCHEMA


def predict_proba(records: List[dict]) -> np.ndarray:
    """Возвращает массив вероятностей для списка объектов."""
    if _MODEL is None:
        raise RuntimeError('Модель не загружена: вызовите load_model() при старте')
    df = pd.DataFrame(records)
    #Гарантируем порядок колонок
    df = df[RAW_FEATURES]
    return _MODEL.predict_proba(df)[:, 1]


def predict_decision(records: List[dict]) -> List[int]:
    """Возвращает решения 0/1 по порогу."""
    proba = predict_proba(records)
    return (proba >= _THRESHOLD).astype(int).tolist()

Overwriting src/predict.py


### **Проверка модулей**

In [12]:
import importlib, sys
from pathlib import Path

#Перезагружаем, если модули уже были импортированы
for mod in ['src.features', 'src.train', 'src.predict']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

import pandas as pd
import numpy as np

from src.features import RAW_FEATURES, add_features, full_transform
from src.train import train_model, evaluate
from src.predict import load_model, predict_proba, predict_decision

#Загружаем данные
df = pd.read_csv('data/raw.csv')
print(f'Данные: {df.shape}')

#Тренировочная выборка (стратифицированная, 80/20 — как в ЛР1)
from sklearn.model_selection import train_test_split
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df['machine_failure'],
    random_state=42,
)
X_train = df.loc[train_idx, RAW_FEATURES]
y_train = df.loc[train_idx, 'machine_failure']
X_test = df.loc[test_idx, RAW_FEATURES]
y_test = df.loc[test_idx, 'machine_failure']
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

#Обучаем простую модель для проверки модулей
params_dummy = {
    'n_estimators': 100,
    'learning_rate': 0.05,
    'num_leaves': 16,
    'class_weight': 'balanced',
}
pipe = train_model(X_train, y_train, params_dummy)
metrics = evaluate(pipe, X_test, y_test)
print('\nМетрики dummy-модели (только для проверки модулей):')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

#Тестируем предсказание из модуля (без сохранения)
import joblib
joblib.dump(pipe, 'models/_test_pipeline.joblib')
import json
schema = {
    col: ({'type': 'number', 'min': float(df[col].min()), 'max': float(df[col].max())}
          if pd.api.types.is_numeric_dtype(df[col])
          else {'type': 'category', 'values': sorted(df[col].dropna().astype(str).unique().tolist())})
    for col in RAW_FEATURES
}
with open('models/_test_schema.json', 'w', encoding='utf-8') as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

load_model('models/_test_pipeline.joblib', 'models/_test_schema.json', threshold=0.022)
sample = df.loc[test_idx[:5], RAW_FEATURES].to_dict('records')
proba = predict_proba(sample)
decisions = predict_decision(sample)
print(f'\nПример предсказания:')
for i, (p, d) in enumerate(zip(proba, decisions)):
    print(f'  {i}: P(failure)={p:.3f} → {"ОСТАНОВИТЬ" if d else "работать"}')

#Убираем временные файлы
import os
os.remove('models/_test_pipeline.joblib')
os.remove('models/_test_schema.json')
print('\nМодули работают корректно')

Данные: (10000, 12)
Train: 8000, Test: 2000

Метрики dummy-модели (только для проверки модулей):
  pr_auc: 0.9059
  roc_auc: 0.9851
  f1: 0.6897
  precision: 0.5660
  recall: 0.8824

Пример предсказания:
  0: P(failure)=0.037 → ОСТАНОВИТЬ
  1: P(failure)=0.006 → работать
  2: P(failure)=0.683 → ОСТАНОВИТЬ
  3: P(failure)=0.004 → работать
  4: P(failure)=0.030 → ОСТАНОВИТЬ

Модули работают корректно


## **Шаг 3 — MLflow: трекинг и реестр моделей**

### **Инициализация MLflow**

In [3]:
import mlflow
import mlflow.sklearn
import os

MLFLOW_URI = f'file:{os.path.abspath("mlruns")}'
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('lab2_variant5_equipment_failure')

print('MLflow tracking URI:', MLFLOW_URI)
print('Эксперимент: lab2_variant5_equipment_failure')

MLflow tracking URI: file:/content/drive/MyDrive/lab2_variant5/mlruns
Эксперимент: lab2_variant5_equipment_failure


### **Обучение + логирование**

In [6]:
import hashlib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score

from src.features import RAW_FEATURES, TARGET
from src.train import train_model, evaluate

#1. Данные
df = pd.read_csv('data/raw.csv')
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df[TARGET], random_state=42,
)
X_train = df.loc[train_idx, RAW_FEATURES]
y_train = df.loc[train_idx, TARGET]
X_test = df.loc[test_idx, RAW_FEATURES]
y_test = df.loc[test_idx, TARGET]

#2. Хеш данных
data_hash = hashlib.md5(open('data/raw.csv', 'rb').read()).hexdigest()[:12]
print(f'data_hash: {data_hash}')

#3. Гиперпараметры из ЛР1
best_params = {
    'n_estimators': 400,
    'learning_rate': 0.010681331534503197,
    'num_leaves': 16,
    'min_child_samples': 6,
    'subsample': 0.9285122106049933,
    'colsample_bytree': 0.7010643539195569,
    'reg_lambda': 2.3564043113674282,
    'class_weight': 'balanced',
    'subsample_freq': 1,
}
THRESHOLD = 0.022

#4. Обучение + логирование
with mlflow.start_run(run_name='lgbm-final-lr1-params') as run:
    pipe = train_model(X_train, y_train, best_params)
    metrics = evaluate(pipe, X_test, y_test)

    mlflow.log_params(best_params)
    mlflow.log_param('threshold', THRESHOLD)
    mlflow.log_param('n_train', len(X_train))
    mlflow.log_param('n_test', len(X_test))

    for k, v in metrics.items():
        mlflow.log_metric(f'{k}_test', v)
        print(f'{k}_test: {v:.4f}')

    mlflow.set_tag('data_version', data_hash)
    mlflow.set_tag('variant', '5_equipment_failure')

    mlflow.sklearn.log_model(pipe, artifact_path='model')

    run_id = run.info.run_id
    print(f'\nRun ID: {run_id}')

data_hash: eca59c0515c0
pr_auc_test: 0.9100
roc_auc_test: 0.9864
f1_test: 0.6740
precision_test: 0.5398
recall_test: 0.8971


2026/10/21 13:11:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Run ID: e506af42e6b342b19fb2c5d31078eb9c


In [5]:
import mlflow

for old_run in ['1b60c90d956443ae9cf9a549950ee258',
                'c7fd5a7d4be249afb34e47cd2a3e701f']:
    try:
        mlflow.delete_run(old_run)
        print(f'Удалён: {old_run}')
    except Exception as e:
        print(f'Не удалось удалить {old_run}: {e}')

Удалён: 1b60c90d956443ae9cf9a549950ee258
Удалён: c7fd5a7d4be249afb34e47cd2a3e701f


### **Регистрация модели**

In [7]:
import mlflow

model_uri = f'runs:/{run_id}/model'
registered = mlflow.register_model(
    model_uri=model_uri,
    name='lab2_equipment_failure',
)
print('Registered model:')
print(f'  name:    {registered.name}')
print(f'  version: {registered.version}')
print(f'  run_id:  {registered.run_id}')

Registered model:
  name:    lab2_equipment_failure
  version: 1
  run_id:  e506af42e6b342b19fb2c5d31078eb9c


Successfully registered model 'lab2_equipment_failure'.
Created version '1' of model 'lab2_equipment_failure'.


### **История запусков**

In [8]:
import mlflow

runs = mlflow.search_runs(order_by=['metrics.pr_auc_test DESC'])
print('Запуски в эксперименте:')
cols = ['run_id', 'tags.mlflow.runName',
        'metrics.pr_auc_test', 'metrics.roc_auc_test',
        'params.n_estimators', 'params.learning_rate',
        'tags.data_version']
available = [c for c in cols if c in runs.columns]
print(runs[available].to_string(index=False))

Запуски в эксперименте:
                          run_id   tags.mlflow.runName  metrics.pr_auc_test  metrics.roc_auc_test params.n_estimators params.learning_rate tags.data_version
e506af42e6b342b19fb2c5d31078eb9c lgbm-final-lr1-params             0.910007              0.986352                 400 0.010681331534503197      eca59c0515c0
33e5d0e82d18404f9bb7ae67326737b8 lgbm-final-lr1-params             0.910007              0.986352                 400 0.010681331534503197      eca59c0515c0
74e1fc3b33ad4ffd81069d5e2684332c     puzzled-squid-729                  NaN                   NaN                None                 None              None


### **Сохранение модели и схемы**

In [9]:
import mlflow.sklearn
import json
import joblib
import pandas as pd
import os
import numpy as np

model = mlflow.sklearn.load_model(f'models:/lab2_equipment_failure/{registered.version}')

os.makedirs('models', exist_ok=True)
joblib.dump(model, 'models/model.joblib')
print('Модель сохранена: models/model.joblib')

df = pd.read_csv('data/raw.csv')
schema = {
    col: (
        {'type': 'number', 'min': float(df[col].min()), 'max': float(df[col].max())}
        if pd.api.types.is_numeric_dtype(df[col])
        else {'type': 'category',
              'values': sorted(df[col].dropna().astype(str).unique().tolist())}
    )
    for col in RAW_FEATURES
}
with open('models/input_schema.json', 'w', encoding='utf-8') as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)
print('Схема сохранена: models/input_schema.json')

loaded = joblib.load('models/model.joblib')
sample = df[RAW_FEATURES].head(3)
assert np.allclose(loaded.predict_proba(sample), model.predict_proba(sample))
print('\nМодель загружается и даёт те же прогнозы')

Модель сохранена: models/model.joblib
Схема сохранена: models/input_schema.json

Модель загружается и даёт те же прогнозы


In [10]:
import mlflow

for old_run in ['33e5d0e82d18404f9bb7ae67326737b8',
                '74e1fc3b33ad4ffd81069d5e2684332c']:
    try:
        mlflow.delete_run(old_run)
        print(f'Удалён: {old_run}')
    except Exception as e:
        print(f'Не удалось удалить {old_run}: {e}')

#Проверка
runs = mlflow.search_runs(order_by=['metrics.pr_auc_test DESC'])
print('\nОсталось запусков:', len(runs))
print(runs[['run_id', 'tags.mlflow.runName', 'metrics.pr_auc_test']].to_string(index=False))

Удалён: 33e5d0e82d18404f9bb7ae67326737b8
Удалён: 74e1fc3b33ad4ffd81069d5e2684332c

Осталось запусков: 1
                          run_id   tags.mlflow.runName  metrics.pr_auc_test
e506af42e6b342b19fb2c5d31078eb9c lgbm-final-lr1-params             0.910007


## **Шаг 4 — FastAPI-сервис**

В сервисе 4 эндпоинта:
*   GET /health — {"status": "ok"}
*   GET /model-info — имя модели, версия, порог, признаки
*   POST /predict — один объект
*   POST /predict-batch — список объектов

Валидация через Pydantic — диапазоны и типы из input_schema.json.

### **Создание src/service.py**

In [11]:
%%writefile src/service.py
"""FastAPI-сервис для прогноза отказа оборудования (ЛР2, вариант 5)."""
import os
import json
from contextlib import asynccontextmanager
from typing import List, Literal

import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from src.predict import load_model, predict_proba, get_threshold, get_schema
from src.features import RAW_FEATURES


#Pydantic-схема входа

class MachineInput(BaseModel):
    type: Literal['L', 'M', 'H'] = Field(..., description='Тип станка')
    air_temperature: float = Field(..., ge=295, le=305,
                                   description='Темп. воздуха, К')
    process_temperature: float = Field(..., ge=305, le=315,
                                       description='Темп. процесса, К')
    rotational_speed: float = Field(..., ge=1000, le=2900,
                                    description='Скорость, об/мин')
    torque: float = Field(..., ge=0, le=80,
                          description='Момент, Н·м')
    tool_wear: float = Field(..., ge=0, le=260,
                             description='Износ, мин')

    model_config = {'extra': 'forbid'}   #запрещаем лишние поля


class PredictionResponse(BaseModel):
    proba: float
    decision: int
    threshold: float


class BatchResponse(BaseModel):
    proba: List[float]
    decision: List[int]
    threshold: float

#Lifespan: загрузка модели при старте
@asynccontextmanager
async def lifespan(app: FastAPI):
    #Модель загружается ОДИН раз при старте сервиса
    model_path = os.environ.get('MODEL_PATH', 'models/model.joblib')
    schema_path = os.environ.get('SCHEMA_PATH', 'models/input_schema.json')
    threshold = float(os.environ.get('THRESHOLD', '0.022'))
    load_model(model_path, schema_path, threshold)
    print(f'[startup] model loaded from {model_path}, threshold={threshold}')
    yield


app = FastAPI(
    title='Equipment Failure Prediction (LR2, variant 5)',
    version='1.0.0',
    lifespan=lifespan,
)

#Эндпоинты
@app.get('/health')
def health():
    return {'status': 'ok'}


@app.get('/model-info')
def model_info():
    return {
        'model_name': 'lab2_equipment_failure',
        'version': '1',
        'threshold': get_threshold(),
        'features': RAW_FEATURES,
        'positive_class': 'machine_failure',
    }


@app.post('/predict', response_model=PredictionResponse)
def predict_one(item: MachineInput):
    record = item.model_dump()
    proba = float(predict_proba([record])[0])
    return PredictionResponse(
        proba=proba,
        decision=int(proba >= get_threshold()),
        threshold=get_threshold(),
    )


@app.post('/predict-batch', response_model=BatchResponse)
def predict_batch(items: List[MachineInput]):
    if not items:
        raise HTTPException(status_code=400, detail='empty batch')
    records = [it.model_dump() for it in items]
    proba = predict_proba(records)
    decisions = (proba >= get_threshold()).astype(int).tolist()
    return BatchResponse(
        proba=proba.tolist(),
        decision=decisions,
        threshold=get_threshold(),
    )

Writing src/service.py


### **Создание src/__init__.py (чтобы работали импорты)**

Без `__init__.py` Python может не найти src.service при запуске uvicorn.

In [12]:
%%writefile src/__init__.py
"""Пакет src для проекта ЛР2."""

Writing src/__init__.py


### **Запуск сервиса в фоне**

In [13]:
import subprocess
import sys
import time
import os
import requests

#Останавливаем старые процессы uvicorn (если были)
subprocess.run(['pkill', '-f', 'uvicorn'], capture_output=True)
time.sleep(1)

#Запускаем uvicorn в фоне
env = os.environ.copy()
env['MODEL_PATH'] = 'models/model.joblib'
env['SCHEMA_PATH'] = 'models/input_schema.json'
env['THRESHOLD'] = '0.022'

log = open('reports/uvicorn.log', 'w')
proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'src.service:app',
     '--host', '127.0.0.1', '--port', '8000', '--log-level', 'info'],
    stdout=log, stderr=log, env=env,
)
print(f'uvicorn PID: {proc.pid}')

#Ждём, пока сервис поднимется (макс 15 сек)
for i in range(30):
    time.sleep(0.5)
    try:
        r = requests.get('http://127.0.0.1:8000/health', timeout=1)
        if r.status_code == 200:
            print(f'Сервис поднялся за {i * 0.5:.1f} сек')
            print('  /health:', r.json())
            break
    except requests.ConnectionError:
        continue
else:
    print('Сервис не поднялся за 15 сек, смотрим лог:')
    log.close()
    with open('reports/uvicorn.log') as f:
        print(f.read()[-2000:])

uvicorn PID: 38112
Сервис поднялся за 3.5 сек
  /health: {'status': 'ok'}


### **Проверка всех эндпоинтов**

In [14]:
import requests

BASE = 'http://127.0.0.1:8000'

#1. /health
print('1) /health:')
print('  ', requests.get(f'{BASE}/health').json())

#2. /model-info
print('\n2) /model-info:')
info = requests.get(f'{BASE}/model-info').json()
print('  ', info)

#3. /predict — корректный запрос
print('\n3) /predict (корректный):')
payload = {
    'type': 'L',
    'air_temperature': 300.0,
    'process_temperature': 310.0,
    'rotational_speed': 1500.0,
    'torque': 40.0,
    'tool_wear': 100.0,
}
r = requests.post(f'{BASE}/predict', json=payload)
print(f'  status={r.status_code}, body={r.json()}')

#4. /predict — неверный тип
print('\n4) /predict (неверный тип type):')
bad = {**payload, 'type': 'X'}
r = requests.post(f'{BASE}/predict', json=bad)
print(f'  status={r.status_code} (ожидаем 422)')

#5. /predict — пропущено поле
print('\n5) /predict (пропущен tool_wear):')
bad = {k: v for k, v in payload.items() if k != 'tool_wear'}
r = requests.post(f'{BASE}/predict', json=bad)
print(f'  status={r.status_code} (ожидаем 422)')

#6. /predict — выход за диапазон
print('\n6) /predict (tool_wear = -5):')
bad = {**payload, 'tool_wear': -5}
r = requests.post(f'{BASE}/predict', json=bad)
print(f'  status={r.status_code} (ожидаем 422)')

#7. /predict-batch — 1000 объектов
print('\n7) /predict-batch (1000 объектов):')
r = requests.post(f'{BASE}/predict-batch', json=[payload] * 1000)
print(f'  status={r.status_code}, len={len(r.json()["proba"])} (ожидаем 200, 1000)')

1) /health:
   {'status': 'ok'}

2) /model-info:
   {'model_name': 'lab2_equipment_failure', 'version': '1', 'threshold': 0.022, 'features': ['type', 'air_temperature', 'process_temperature', 'rotational_speed', 'torque', 'tool_wear'], 'positive_class': 'machine_failure'}

3) /predict (корректный):
  status=200, body={'proba': 0.011117928174522779, 'decision': 0, 'threshold': 0.022}

4) /predict (неверный тип type):
  status=422 (ожидаем 422)

5) /predict (пропущен tool_wear):
  status=422 (ожидаем 422)

6) /predict (tool_wear = -5):
  status=422 (ожидаем 422)

7) /predict-batch (1000 объектов):
  status=200, len=1000 (ожидаем 200, 1000)


In [15]:
import requests
print(requests.get('http://127.0.0.1:8000/health').json())

{'status': 'ok'}


## **Шаг 5 — Тесты**

### **Создание tests/test_service.py**

In [16]:
%%writefile tests/test_service.py
"""Тесты FastAPI-сервиса (ЛР2, вариант 5)."""
import pytest
from fastapi.testclient import TestClient

from src.service import app
from src.predict import load_model


@pytest.fixture(scope='module')
def client():
    load_model('models/model.joblib', 'models/input_schema.json', 0.022)
    return TestClient(app)


VALID = {
    'type': 'L',
    'air_temperature': 300.0,
    'process_temperature': 310.0,
    'rotational_speed': 1500.0,
    'torque': 40.0,
    'tool_wear': 100.0,
}


def test_health(client):
    r = client.get('/health')
    assert r.status_code == 200
    assert r.json()['status'] == 'ok'


def test_predict_ok(client):
    r = client.post('/predict', json=VALID)
    assert r.status_code == 200
    body = r.json()
    assert 0.0 <= body['proba'] <= 1.0
    assert body['decision'] in (0, 1)
    assert body['threshold'] == 0.022


def test_predict_wrong_type(client):
    bad = {**VALID, 'type': 'X'}
    r = client.post('/predict', json=bad)
    assert r.status_code == 422


def test_predict_missing_field(client):
    bad = {k: v for k, v in VALID.items() if k != 'tool_wear'}
    r = client.post('/predict', json=bad)
    assert r.status_code == 422


def test_predict_out_of_range(client):
    bad = {**VALID, 'tool_wear': -5}
    r = client.post('/predict', json=bad)
    assert r.status_code == 422


def test_predict_batch_1000(client):
    r = client.post('/predict-batch', json=[VALID] * 1000)
    assert r.status_code == 200
    body = r.json()
    assert len(body['proba']) == 1000
    assert len(body['decision']) == 1000
    assert all(0.0 <= p <= 1.0 for p in body['proba'])


def test_predict_batch_empty(client):
    r = client.post('/predict-batch', json=[])
    assert r.status_code == 400


def test_model_info(client):
    r = client.get('/model-info')
    assert r.status_code == 200
    body = r.json()
    assert body['model_name'] == 'lab2_equipment_failure'
    assert body['threshold'] == 0.022
    assert len(body['features']) == 6

Writing tests/test_service.py


### **Создание tests/__init__.py**

In [17]:
%%writefile tests/__init__.py

Writing tests/__init__.py


### **Запуск тестов через subprocess**

In [18]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('--- STDERR ---')
    print(result.stderr)
    raise RuntimeError(f'pytest failed with code {result.returncode}')
print('Все тесты прошли')

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/lab2_variant5
plugins: locust-2.46.7, Faker-40.43.0, langsmith-0.14.11, typeguard-4.6.0, platformdirs-4.13.4, anyio-4.15.1
collecting ... collected 8 items

tests/test_service.py::test_health PASSED                                [ 12%]
tests/test_service.py::test_predict_ok PASSED                            [ 25%]
tests/test_service.py::test_predict_wrong_type PASSED                    [ 37%]
tests/test_service.py::test_predict_missing_field PASSED                 [ 50%]
tests/test_service.py::test_predict_out_of_range PASSED                  [ 62%]
tests/test_service.py::test_predict_batch_1000 PASSED                    [ 75%]
tests/test_service.py::test_predict_batch_empty PASSED                   [ 87%]
tests/test_service.py::test_model_info PASSED                    

## **Шаг 6 — Нагрузочный тест (Locust)**

### **Создание locustfile.py**

In [19]:
%%writefile locustfile.py
"""Нагрузочный сценарий для ЛР2, вариант 5.

Смешанный профиль:
  - 80% — одиночные /predict (потоковый режим, каждую минуту)
  - 20% — батч /predict-batch (на 50 объектов)

Требование варианта 5: p95 ≤ 200 мс для одиночного запроса.
"""
from locust import HttpUser, task, between


SINGLE = {
    'type': 'L',
    'air_temperature': 300.0,
    'process_temperature': 310.0,
    'rotational_speed': 1500.0,
    'torque': 40.0,
    'tool_wear': 100.0,
}

BATCH_50 = [SINGLE] * 50


class ScoringUser(HttpUser):
    wait_time = between(0.01, 0.05)   #имитация потока

    @task(8)
    def predict_one(self):
        self.client.post('/predict', json=SINGLE)

    @task(2)
    def predict_batch_50(self):
        self.client.post('/predict-batch', json=BATCH_50)

Writing locustfile.py


### **Запуск Locust через subprocess (5, 20, 50 пользователей)**

In [20]:
import subprocess
import pandas as pd
import time
import os

os.makedirs('reports', exist_ok=True)

#Проверим, что сервис жив
import requests
try:
    assert requests.get('http://127.0.0.1:8000/health', timeout=2).json()['status'] == 'ok'
    print('Сервис жив')
except Exception as e:
    raise RuntimeError(f'Сервис недоступен: {e}')

results = []
for users in [5, 20, 50]:
    csv_prefix = f'reports/load_{users}'
    print(f'\n=== {users} пользователей ===')
    cmd = [
        'locust', '-f', 'locustfile.py',
        '--headless',
        '-u', str(users), '-r', '5', '-t', '30s',
        '--host', 'http://127.0.0.1:8000',
        '--csv', csv_prefix,
        '--only-summary',
    ]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1000:])
        continue
    #Читаем CSV с агрегированными метриками
    stats = pd.read_csv(f'{csv_prefix}_stats.csv')
    stats = stats[stats['Name'] != 'Aggregated']   #оставим только эндпоинты
    aggregated = pd.read_csv(f'{csv_prefix}_stats.csv')
    aggregated = aggregated[aggregated['Name'] == 'Aggregated'].iloc[0]
    results.append({
        'users': users,
        'requests': int(aggregated['Request Count']),
        'failures': int(aggregated['Failure Count']),
        'rps': float(aggregated['Requests/s']),
        'p50_ms': float(aggregated['50%']),
        'p95_ms': float(aggregated['95%']),
        'p99_ms': float(aggregated['99%']),
    })

df_results = pd.DataFrame(results)
print('\n=== Итоги нагрузки ===')
print(df_results.to_string(index=False))

#Проверка требования p95 ≤ 200 мс
for _, row in df_results.iterrows():
    status = 'Работает' if row['p95_ms'] <= 200 else 'Не работает'
    print(f"{status} {int(row['users'])} польз.: p95 = {row['p95_ms']:.1f} мс "
          f"(требование ≤ 200 мс)")

df_results.to_csv('reports/load_summary.csv', index=False)
print('\nСохранено: reports/load_summary.csv')

Сервис жив

=== 5 пользователей ===

=== 20 пользователей ===

=== 50 пользователей ===

=== Итоги нагрузки ===
 users  requests  failures        rps  p50_ms  p95_ms  p99_ms
     5      2562         0  87.554202    24.0    49.0    61.0
    20      3374         0 114.456441   130.0   220.0   260.0
    50      3081         0 104.256541   380.0   620.0   700.0
Работает 5 польз.: p95 = 49.0 мс (требование ≤ 200 мс)
Не работает 20 польз.: p95 = 220.0 мс (требование ≤ 200 мс)
Не работает 50 польз.: p95 = 620.0 мс (требование ≤ 200 мс)

Сохранено: reports/load_summary.csv


### **Вывод:**

**Требование p95 ≤ 200 мс** выполняется при 5 пользователях, но не при 20+
Это нормально для Colab — разделяемая среда, нет выделенных ресурсов.

При 5 пользователях (имитация потокового режима — один станок раз в минуту, но с запасом) p95 = 49 мс, что в 4 раза ниже требования (≤ 200 мс). При 20 и 50 пользователях p95 превышает 200 мс из-за ограничений Colab: среда разделяемая, один vCPU, нет выделенного сервера. На выделенном сервере ожидаемая пропускная способность — сотни запросов в секунду при p95 < 200 мс.

**Ключевой вывод:** требование p95 ≤ 200 мс достигается уже при 5 конкурентных пользователях (87 RPS), что с запасом покрывает потоковый режим варианта `(1 станок раз в минуту × N станков, где N ≈ 100–1000).`

Один пакетный запрос на 1000 объектов обрабатывается за 30 мс (33 294 req/s), тогда как 1000 одиночных запросов — за 11 044 мс (90.5 req/s). Ускорение — 367× — это следствие (а) накладных расходов HTTP на каждый запрос, (б) векторизации LightGBM при предсказании на массиве. Для потокового режима это означает, что батчинг 1000 записей раз в минуту — тривиально лёгкая задача, а одиночные запросы — тоже укладываются в требование.

Точка насыщения — около 20 конкурентных пользователей (RPS ≈ 115). После неё пропускная способность падает, а задержки растут экспоненциально, что характерно для системы в состоянии перегрузки. Для варианта 5 (потоковый режим, 1 станок раз в минуту) требуется ~0.017 RPS на станок — 1000 станков дают 16.7 RPS, что в 7 раз меньше точки насыщения (115 RPS). Сервис с запасом покрывает производственную нагрузку.

### **Сравнение одиночного и пакетного режимов**

In [21]:
import requests
import time
import numpy as np

BASE = 'http://127.0.0.1:8000'
SINGLE = {
    'type': 'L', 'air_temperature': 300.0, 'process_temperature': 310.0,
    'rotational_speed': 1500.0, 'torque': 40.0, 'tool_wear': 100.0,
}

#1. 1000 одиночных запросов
print('Измерение: 1000 одиночных /predict...')
times_single = []
for _ in range(1000):
    t0 = time.perf_counter()
    requests.post(f'{BASE}/predict', json=SINGLE)
    times_single.append((time.perf_counter() - t0) * 1000)

#2. 1 пакетный запрос на 1000 объектов
print('Измерение: 1 пакетный /predict-batch на 1000...')
t0 = time.perf_counter()
requests.post(f'{BASE}/predict-batch', json=[SINGLE] * 1000)
time_batch = (time.perf_counter() - t0) * 1000

times_single = np.array(times_single)
print()
print(f'1000 одиночных:')
print(f'  всего:       {times_single.sum():.0f} мс')
print(f'  p50:         {np.percentile(times_single, 50):.1f} мс')
print(f'  p95:         {np.percentile(times_single, 95):.1f} мс')
print(f'  среднее:     {times_single.mean():.1f} мс')
print(f'  throughput:  {1000 / (times_single.sum() / 1000):.1f} req/s')
print()
print(f'1 пакетный на 1000:')
print(f'  всего:       {time_batch:.1f} мс')
print(f'  throughput:  {1000 / (time_batch / 1000):.1f} req/s')
print()
print(f'Ускорение пакетного режима: {times_single.sum() / time_batch:.1f}×')

Измерение: 1000 одиночных /predict...
Измерение: 1 пакетный /predict-batch на 1000...

1000 одиночных:
  всего:       11044 мс
  p50:         10.0 мс
  p95:         15.5 мс
  среднее:     11.0 мс
  throughput:  90.5 req/s

1 пакетный на 1000:
  всего:       30.0 мс
  throughput:  33294.4 req/s

Ускорение пакетного режима: 367.7×


## **Шаг 7 — Дрейф данных (смещение температуры процесса)**

**Стратегия:**

1. **Эталон:** тестовая выборка из ЛР1 (2000 объектов, несдвинутые данные).

2. **Новые данные:** та же выборка с правдоподобным сдвигом:

    - process_temperature + 3 K (перегрев — типичная причина отказов).

    - air_temperature + 1 K (следствие перегрева среды).

    - tool_wear × 1.15 (износ ускоряется при перегреве).

3. **Проверка:** Evidently → отчёт о дрейфе → PR-AUC на сдвинутых данных.

In [22]:
import select
print('epoll доступен:', hasattr(select, 'epoll'))

epoll доступен: True


In [23]:
import requests
print(requests.get('http://127.0.0.1:8000/health').json())

{'status': 'ok'}


### **Генерация сдвинутых данных**

In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from src.features import RAW_FEATURES, TARGET

#1. Загружаем данные и восстанавливаем тестовую выборку
df = pd.read_csv('data/raw.csv')
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df[TARGET], random_state=42,
)
X_test = df.loc[test_idx, RAW_FEATURES].copy()
y_test = df.loc[test_idx, TARGET].copy()

print(f'Reference (test): {X_test.shape}, отказов: {y_test.sum()} ({y_test.mean()*100:.2f}%)')

#2. Смещение: перегрев процесса + ускоренный износ
np.random.seed(42)
X_drift = X_test.copy()
X_drift['process_temperature'] = X_drift['process_temperature'] + 3.0
X_drift['air_temperature']     = X_drift['air_temperature']     + 1.0
X_drift['tool_wear']           = X_drift['tool_wear'] * 1.15

print('\nЧто изменилось (средние):')
for col in ['air_temperature', 'process_temperature', 'tool_wear']:
    old = X_test[col].mean()
    new = X_drift[col].mean()
    pct = (new / old - 1) * 100
    print(f'  {col:22}: {old:.2f} → {new:.2f}  ({pct:+.1f}%)')

#Сохраним сдвинутые данные для отчёта
X_drift.to_csv('data/test_drifted.csv', index=False)
print('\nСохранено: data/test_drifted.csv')

Reference (test): (2000, 6), отказов: 68 (3.40%)

Что изменилось (средние):
  air_temperature       : 300.00 → 301.00  (+0.3%)
  process_temperature   : 310.00 → 313.00  (+1.0%)
  tool_wear             : 109.02 → 125.37  (+15.0%)

Сохранено: data/test_drifted.csv


### **Отчёт Evidently**

In [25]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset
from evidently.metrics import DatasetDriftMetric

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

#Создаём отчёт
report = Report(metrics=[
    DataDriftPreset(),
    DatasetDriftMetric(),
])

report.run(reference_data=X_test, current_data=X_drift)
report.save_html('reports/drift_report.html')
print('Отчёт сохранён: reports/drift_report.html')

#Извлекаем результаты
res = report.as_dict()

#Доля сдвинувшихся признаков
drift_summary = None
for m in res['metrics']:
    if m['metric'] == 'DatasetDriftMetric':
        drift_summary = m['result']
        break

if drift_summary:
    print()
    print(f"Всего признаков:      {drift_summary['number_of_columns']}")
    print(f"Сдвинулось признаков: {drift_summary['number_of_drifted_columns']}")
    print(f"Доля сдвинувшихся:    {drift_summary['share_of_drifted_columns']:.2%}")
    print(f"Дрейф датасета:       {drift_summary['dataset_drift']}")

Отчёт сохранён: reports/drift_report.html

Всего признаков:      6
Сдвинулось признаков: 3
Доля сдвинувшихся:    50.00%
Дрейф датасета:       True


### **Оценка качества на сдвинутых данных**

In [26]:
import mlflow.sklearn
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score

#Загружаем финальную модель
model = mlflow.sklearn.load_model('models:/lab2_equipment_failure/1')

#Предсказания на reference и drift
proba_ref   = model.predict_proba(X_test)[:, 1]
proba_drift = model.predict_proba(X_drift)[:, 1]

#Метрики
ap_ref   = average_precision_score(y_test, proba_ref)
ap_drift = average_precision_score(y_test, proba_drift)
auc_ref  = roc_auc_score(y_test, proba_ref)
auc_drift = roc_auc_score(y_test, proba_drift)

print(f'{"Метрика":12} {"reference":>12} {"drift":>12} {"Δ":>10}')
print('-' * 50)
print(f'{"PR-AUC":12} {ap_ref:>12.4f} {ap_drift:>12.4f} {ap_drift - ap_ref:>+10.4f}')
print(f'{"ROC-AUC":12} {auc_ref:>12.4f} {auc_drift:>12.4f} {auc_drift - auc_ref:>+10.4f}')
print()

delta = ap_drift - ap_ref
if abs(delta) < 0.03:
    print(f'Падение PR-AUC в пределах шума (|Δ| < 0.03)')
else:
    print(f'Падение PR-AUC значимо (Δ = {delta:+.4f})')

#Сколько объектов "переключилось"
threshold = 0.022
dec_ref   = (proba_ref   >= threshold).astype(int)
dec_drift = (proba_drift >= threshold).astype(int)
changed = (dec_ref != dec_drift).sum()
print(f'\nРешения изменились у {changed} из {len(y_test)} объектов '
      f'({changed/len(y_test)*100:.1f}%)')

Метрика         reference        drift          Δ
--------------------------------------------------
PR-AUC             0.9100       0.4848    -0.4252
ROC-AUC            0.9864       0.9143    -0.0721

Падение PR-AUC значимо (Δ = -0.4252)

Решения изменились у 351 из 2000 объектов (17.5%)


### **Только температура**

In [27]:
#Сценарий «только температура процесса» — как в методичке
X_drift_only_temp = X_test.copy()
X_drift_only_temp['process_temperature'] = X_drift_only_temp['process_temperature'] + 3.0
#Без air_temperature и tool_wear

proba_drift_t = model.predict_proba(X_drift_only_temp)[:, 1]
ap_drift_t = average_precision_score(y_test, proba_drift_t)
auc_drift_t = roc_auc_score(y_test, proba_drift_t)

print('Сценарий «только process_temperature + 3K»:')
print(f'  PR-AUC:  {ap_drift_t:.4f} (Δ={ap_drift_t - ap_ref:+.4f})')
print(f'  ROC-AUC: {auc_drift_t:.4f} (Δ={auc_drift_t - auc_ref:+.4f})')

Сценарий «только process_temperature + 3K»:
  PR-AUC:  0.5962 (Δ=-0.3138)
  ROC-AUC: 0.9498 (Δ=-0.0366)


### **Правило переобучения**

In [28]:
print('''
╔══════════════════════════════════════════════════════════════════╗
║            ПРАВИЛО ЗАПУСКА ПЕРЕОБУЧЕНИЯ             ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  УСЛОВИЕ (любое из двух):                                        ║
║    1) Доля признаков со значимым сдвигом (p < 0.05 по KS или     ║
║       chi²) превышает 30% на окне последних 7 дней.              ║
║    2) PR-AUC на размеченной выборке падает более чем на 0.05     ║
║       относительно приёмочного уровня (0.910 из ЛР1).           ║
║                                                                  ║
║  ПЕРИОДИЧНОСТЬ ПРОВЕРКИ:                                         ║
║    • Evidently/KS-тест — ежедневно (автоматически).              ║
║    • PR-AUC — по мере поступления разметки (раз в месяц).         ║
║    • Полный отчёт — еженедельно, на совещании эксплуатации.      ║
║                                                                  ║
║  ДЕЙСТВИЕ:                                                       ║
║    • Переобучение на данных за последние 6 месяцев.              ║
║    • Сравнение новой модели с текущей на отложенной выборке      ║
║      (свежие 20% данных).                                        ║
║    • Замена модели — только если PR-AUC вырос ≥ 0.03.           ║
║    • Откат — если новая модель хуже по калибровке (Brier).       ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
''')

print('Проверка правила на нашем сценарии дрейфа:')
print(f'  Доля сдвинувшихся признаков: 50% > 30% → срабатывает')
print(f'  PR-AUC упал с {ap_ref:.4f} до {ap_drift:.4f} (Δ={ap_drift-ap_ref:+.4f})')
if abs(ap_drift - ap_ref) > 0.05:
    print(f'  |Δ| > 0.05 → срабатывает по второму условию')
else:
    print(f'  |Δ| < 0.05 → не срабатывает по второму условию')
print()
print('Вывод: оба условия срабатывают → переобучение обязательно.')


╔══════════════════════════════════════════════════════════════════╗
║            ПРАВИЛО ЗАПУСКА ПЕРЕОБУЧЕНИЯ             ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  УСЛОВИЕ (любое из двух):                                        ║
║    1) Доля признаков со значимым сдвигом (p < 0.05 по KS или     ║
║       chi²) превышает 30% на окне последних 7 дней.              ║
║    2) PR-AUC на размеченной выборке падает более чем на 0.05     ║
║       относительно приёмочного уровня (0.910 из ЛР1).           ║
║                                                                  ║
║  ПЕРИОДИЧНОСТЬ ПРОВЕРКИ:                                         ║
║    • Evidently/KS-тест — ежедневно (автоматически).              ║
║    • PR-AUC — по мере поступления разметки (раз в месяц).         ║
║    • Полный отчёт — еженедельно, на совещании эксплуатации.      ║
║                                             

### **Вывод**

### **Сценарий дрейфа**

В соответствии с методичкой,
построены **два сценария**:

- **A. Изолированный сдвиг (строго по заданию):** `process_temperature + 3 K`.
- **B. Комплексный сценарий (производственная реальность):**
  `process_temperature + 3 K`, `air_temperature + 1 K`,
  `tool_wear × 1.15` (перегрев ускоряет износ).

Сдвиг правдоподобен: такой режим возможен при смене режима работы
станка, износе системы охлаждения или смене сырья.

### **Результаты Evidently (сценарий B)**

- **3 из 6** признаков сдвинулись значимо (KS-тест, p < 0.05).
- **Доля сдвинувшихся = 50 %**, `dataset_drift = True`.
- Отчёт сохранён: `reports/drift_report.html`.

### Влияние на качество модели

| Метрика | Reference | Только T+3K | Комплексный |
|---------|-----------|-------------|-------------|
| **PR-AUC** | 0.9100 | **0.5962** (−0.3138) | **0.4848** (−0.4252) |
| **ROC-AUC** | 0.9864 | 0.9498 (−0.0366) | 0.9143 (−0.0721) |
| Объектов переключилось | — | — | 351 (17.5 %) |

### Интерпретация

1. **Изолированный сдвиг температуры +3 K** (1 % от среднего значения 310 K)
   **снижает PR-AUC на 31.4 п.п.** — модель, обученная на «холодном» режиме,
   не узнаёт «горячий». Это **концептуальный дрейф**, а не просто шум.

2. **ROC-AUC устойчив** (−3.7 п.п. для температуры, −7.2 п.п. для комплексного) —
   модель по-прежнему ранжирует объекты правильно, но **калибровка рушится**:
   при +3 K модель считает «рискованными» слишком много объектов.

3. **Комплексный сценарий** усиливает эффект: PR-AUC падает в 1.9 раза.
   Совместное действие перегрева и ускоренного износа — **самый опасный**
   сценарий для оборудования.

4. **17.5 % объектов «переключилось»** — 351 из 2000. При реальном внедрении
   это означало бы **массовые ложные тревоги** после смены режима.

### **Правило переобучения (проверяемое)**

**Условие (любое из двух):**
1. Доля признаков со значимым сдвигом (p < 0.05 по KS-тесту или χ²)
   превышает **30 %** на окне последних 7 дней.
2. PR-AUC на размеченной выборке падает более чем на **0.05**
   относительно приёмочного уровня (0.910 из ЛР1).

**Периодичность проверки:**
- Evidently / KS-тест — **ежедневно** (автоматически).
- PR-AUC — **по мере поступления разметки** (раз в месяц).
- Полный отчёт — **еженедельно**, на совещании эксплуатации.

**Действие:**
- Переобучение на данных за последние **6 месяцев**.
- Сравнение новой модели с текущей на **отложенной выборке** (свежие 20 % данных).
- Замена модели — только если **PR-AUC вырос ≥ 0.03**.
- Откат — если новая модель хуже по калибровке (Brier score).

### Проверка правила на нашем сценарии

| Условие | Значение | Срабатывает? |
|---------|----------|--------------|
| Доля сдвинувшихся признаков > 30 % | **50 %** | да |
| Падение PR-AUC > 0.05 | **−0.31** (T+3K) / **−0.43** (комплексный) | да |

**Вывод:** оба условия срабатывают → переобучение обязательно.

## **Шаг 8 — Gradio-интерфейс**

### **Создаём аpp.py (Gradio UI)**

In [29]:
%%writefile app.py
"""Gradio-интерфейс для ЛР2, вариант 5."""
import gradio as gr
import requests

API_URL = 'http://127.0.0.1:8000'


def score(machine_type, air_t, proc_t, speed, torque, wear):
    payload = {
        'type': machine_type,
        'air_temperature': float(air_t),
        'process_temperature': float(proc_t),
        'rotational_speed': float(speed),
        'torque': float(torque),
        'tool_wear': float(wear),
    }
    try:
        r = requests.post(f'{API_URL}/predict', json=payload, timeout=5)
    except Exception as e:
        return f'Ошибка соединения: {e}'
    if r.status_code != 200:
        return f'Ошибка валидации ({r.status_code}):\n{r.json()}'
    out = r.json()
    verdict = 'ОСТАНОВИТЬ' if out['decision'] else 'РАбОТАТЬ'
    return (
        f"Вероятность отказа: {out['proba']:.4f}\n"
        f"Порог: {out['threshold']}\n"
        f"Решение: {verdict}"
    )


demo = gr.Interface(
    fn=score,
    inputs=[
        gr.Dropdown(['L', 'M', 'H'], label='Тип станка', value='L'),
        gr.Number(label='Темп. воздуха, К', value=300.0),
        gr.Number(label='Темп. процесса, К', value=310.0),
        gr.Number(label='Скорость, об/мин', value=1500.0),
        gr.Number(label='Момент, Н·м', value=40.0),
        gr.Number(label='Износ, мин', value=100.0),
    ],
    outputs=gr.Textbox(label='Результат', lines=5),
    title='Прогноз отказа оборудования (ЛР2, вариант 5)',
    description='Модель LightGBM (PR-AUC 0.910 на тесте). Сервис FastAPI.',
)

if __name__ == '__main__':
    demo.launch(server_name='0.0.0.0', server_port=7860,
                share=True, show_error=True)

Writing app.py


### **Запуск Gradio через subprocess**

In [30]:
import subprocess
import sys
import time
import os

#Останавливаем старые Gradio
subprocess.run(['pkill', '-f', 'app.py'], capture_output=True)
time.sleep(1)

log = open('reports/gradio.log', 'w')
proc_gradio = subprocess.Popen(
    [sys.executable, 'app.py'],
    stdout=log, stderr=log,
    env=os.environ.copy(),
)
print(f'Gradio PID: {proc_gradio.pid}')

print('Ожидаем публичную ссылку (до 90 сек)...')
public_url = None
for i in range(90):
    time.sleep(1)
    with open('reports/gradio.log') as f:
        log_text = f.read()
    #Gradio 5.x пишет: * Running on public URL: https://xxx.gradio.live
    for line in log_text.splitlines():
        if 'gradio.live' in line and 'Running on public URL' in line:
            public_url = line.split('public URL:')[-1].strip()
            break
    if public_url:
        break
    if i % 15 == 0 and i > 0:
        print(f'  ... ждём {i} сек')

if public_url:
    print(f'\nПубличная ссылка: {public_url}')
    with open('reports/gradio_url.txt', 'w') as f:
        f.write(public_url)
    print('Сохранено: reports/gradio_url.txt')
else:
    print('\n Не дождались публичной ссылки за 90 сек')
    log.close()
    print('ЛОГ')
    print(open('reports/gradio.log').read()[-3000:])

Gradio PID: 45283
Ожидаем публичную ссылку (до 90 сек)...
  ... ждём 15 сек
  ... ждём 30 сек
  ... ждём 45 сек
  ... ждём 60 сек
  ... ждём 75 сек

 Не дождались публичной ссылки за 90 сек
ЛОГ



In [32]:
import os

if os.path.exists('reports/gradio.log'):
    print('=== reports/gradio.log (последние 3000 символов) ===')
    with open('reports/gradio.log') as f:
        print(f.read()[-3000:])
else:
    print('Файл reports/gradio.log не создан')

#Проверим, что процесс жив
import subprocess
r = subprocess.run(['ps', 'aux'], capture_output=True, text=True)
lines = [l for l in r.stdout.splitlines() if 'app.py' in l or 'gradio' in l.lower()]
print('\n=== Процессы Gradio ===')
for l in lines:
    print(l)

=== reports/gradio.log (последние 3000 символов) ===


=== Процессы Gradio ===
root       45283  2.1  1.1 743376 147172 ?       Sl   13:49   0:04 /usr/bin/python3 app.py
root       45309  0.0  0.1 1238804 14604 ?       Sl   13:49   0:00 /root/.cache/huggingface/gradio/frpc/frpc_linux_amd64_v0.3 http -n L3MLiUtslWpjFJ1h5Ux09dRb-KndyG6wRVLjwcxgB1E -l 7860 -i 0.0.0.0 --uc --sd random --ue --server_addr gradio-live.com:7000 --disable_log_color --tls_enable --tls_trusted_ca_file .gradio/certificate.pem


### **Ивлечение ссылки из процесса**

In [33]:
import subprocess
import re

#Находим процесс frpc и извлекаем subdomain
r = subprocess.run(['ps', 'aux'], capture_output=True, text=True)
for line in r.stdout.splitlines():
    if 'frpc' in line and 'gradio-live.com' in line:
        m = re.search(r'-n\s+(\S+)', line)
        if m:
            subdomain = m.group(1)
            public_url = f'https://{subdomain}.gradio.live'
            print(f'Публичная ссылка: {public_url}')
            with open('reports/gradio_url.txt', 'w') as f:
                f.write(public_url)
            print('Сохранено: reports/gradio_url.txt')
            break
else:
    print('frpc-процесс не найден — Gradio не создал туннель')

Публичная ссылка: https://L3MLiUtslWpjFJ1h5Ux09dRb-KndyG6wRVLjwcxgB1E.gradio.live
Сохранено: reports/gradio_url.txt


### **Проверка, что ссылка жива**

In [35]:
import subprocess
import sys
import time
import os

subprocess.run(['pkill', '-9', '-f', 'app.py'], capture_output=True)
subprocess.run(['pkill', '-9', '-f', 'frpc'], capture_output=True)
time.sleep(2)

log = open('reports/gradio.log', 'w')

#Запускаем с PYTHONUNBUFFERED=1, чтобы stdout шёл сразу, без буфера
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

proc = subprocess.Popen(
    [sys.executable, '-u', 'app.py'],   #-u = unbuffered
    stdout=log, stderr=subprocess.STDOUT,   #stderr тоже в лог
    env=env,
)
print(f'Gradio PID: {proc.pid}')
print('Ожидаем ссылку в логе (до 90 сек)...')

public_url = None
for i in range(90):
    time.sleep(1)
    log.flush()
    with open('reports/gradio.log') as f:
        text = f.read()
    #Gradio 5.x пишет: "* Running on public URL: https://xxx.gradio.live"
    for line in text.splitlines():
        if 'Running on public URL' in line:
            public_url = line.split('public URL:')[-1].strip()
            break
    if public_url:
        break
    if i % 15 == 0 and i > 0:
        print(f'  ... ждём {i} сек, лог: {len(text)} символов')

if public_url:
    print(f'\nПубличная ссылка: {public_url}')
    with open('reports/gradio_url.txt', 'w') as f:
        f.write(public_url)
else:
    print('\nСсылка не появилась в логе. Содержимое лога:')
    log.close()
    with open('reports/gradio.log') as f:
        print(f.read())

Gradio PID: 46723
Ожидаем ссылку в логе (до 90 сек)...

Публичная ссылка: https://d3cd4e87efd8f35da1.gradio.live


In [36]:
import requests

url = 'https://d3cd4e87efd8f35da1.gradio.live'
try:
    r = requests.get(url, timeout=15, allow_redirects=True)
    print(f'HTTP {r.status_code}')
    if r.status_code == 200:
        print('Публичная ссылка работает')
except Exception as e:
    print(f'Ошибка: {e}')

HTTP 200
Публичная ссылка работает


### **Проверка функции через публичный URL**

In [38]:
from gradio_client import Client

client = Client('https://d3cd4e87efd8f35da1.gradio.live')

#1. Сначала узнаем, какие API доступны
print('Доступные API:')
for info in client.view_api(return_format='dict')['named_endpoints'].items():
    print(' ', info)

#2. Вызываем с правильным api_name
result = client.predict(
    machine_type='L',
    air_t=300.0,
    proc_t=310.0,
    speed=1500.0,
    torque=40.0,
    wear=100.0,
    api_name='/predict',
)
print('\nРезультат из публичного Gradio:')
print(result)

Loaded as API: https://d3cd4e87efd8f35da1.gradio.live/ ✔
Доступные API:
Client.predict() Usage Info
---------------------------
Named API endpoints: 1

 - predict(machine_type, air_t, proc_t, speed, torque, wear, api_name="/predict") -> результат
    Parameters:
     - [Dropdown] machine_type: Literal['L', 'M', 'H'] (not required, defaults to:   L)  
     - [Number] air_t: float (not required, defaults to:   300.0)  
     - [Number] proc_t: float (not required, defaults to:   310.0)  
     - [Number] speed: float (not required, defaults to:   1500.0)  
     - [Number] torque: float (not required, defaults to:   40.0)  
     - [Number] wear: float (not required, defaults to:   100.0)  
    Returns:
     - [Textbox] результат: str 

  ('/predict', {'parameters': [{'label': 'Тип станка', 'parameter_name': 'machine_type', 'parameter_has_default': True, 'parameter_default': 'L', 'type': {'type': 'string', 'enum': ['L', 'M', 'H']}, 'python_type': {'type': "Literal['L', 'M', 'H']", 'descripti

### **Финальная проверка**

In [39]:
from gradio_client import Client

url = 'https://d3cd4e87efd8f35da1.gradio.live'
client = Client(url)

#Проверим API
apis = list(client.view_api(return_format='dict')['named_endpoints'].keys())
print(f'Gradio {url}')
print(f'Доступные API: {apis}')

#Вызываем функцию
result = client.predict(
    machine_type='L',
    air_t=300.0,
    proc_t=310.0,
    speed=1500.0,
    torque=40.0,
    wear=100.0,
    api_name='/predict',
)
print('\nРезультат:')
print(result)

#Проверим «плохой» пример — модель должна вернуть ОСТАНОВИТЬ при высоком износе
result_bad = client.predict(
    machine_type='L',
    air_t=300.0,
    proc_t=310.0,
    speed=1500.0,
    torque=40.0,
    wear=250.0,   #очень высокий износ
    api_name='/predict',
)
print('\nРезультат при износе 250 мин:')
print(result_bad)

Loaded as API: https://d3cd4e87efd8f35da1.gradio.live/ ✔
Client.predict() Usage Info
---------------------------
Named API endpoints: 1

 - predict(machine_type, air_t, proc_t, speed, torque, wear, api_name="/predict") -> результат
    Parameters:
     - [Dropdown] machine_type: Literal['L', 'M', 'H'] (not required, defaults to:   L)  
     - [Number] air_t: float (not required, defaults to:   300.0)  
     - [Number] proc_t: float (not required, defaults to:   310.0)  
     - [Number] speed: float (not required, defaults to:   1500.0)  
     - [Number] torque: float (not required, defaults to:   40.0)  
     - [Number] wear: float (not required, defaults to:   100.0)  
    Returns:
     - [Textbox] результат: str 

Gradio https://d3cd4e87efd8f35da1.gradio.live
Доступные API: ['/predict']

Результат:
Вероятность отказа: 0.0111
Порог: 0.022
Решение: РАбОТАТЬ

Результат при износе 250 мин:
Вероятность отказа: 0.5584
Порог: 0.022
Решение: ОСТАНОВИТЬ


## **Шаг 9 — Проверка воспроизводимости**

### **Создание  requirements.txt**

In [40]:
import subprocess
import sys

result = subprocess.run([sys.executable, '-m', 'pip', 'freeze'],
                        capture_output=True, text=True)
with open('requirements.txt', 'w') as f:
    f.write(result.stdout)
print(f'requirements.txt: {len(result.stdout.splitlines())} пакетов')

for line in result.stdout.splitlines():
    if any(line.startswith(p) for p in (
        'fastapi', 'uvicorn', 'pydantic', 'mlflow', 'evidently',
        'gradio', 'scikit-learn', 'lightgbm', 'catboost', 'locust',
    )):
        print(' ', line)

requirements.txt: 692 пакетов
  catboost==1.2.10
  evidently==0.4.40
  fastapi==0.115.14
  gradio==5.40.0
  gradio_client==1.11.0
  lightgbm==4.6.0
  locust==2.46.7
  mlflow==2.22.5
  mlflow-skinny==2.22.5
  pydantic==2.11.10
  pydantic_core==2.33.2
  scikit-learn==1.5.2
  uvicorn==0.32.1


### **Проверка загрузки модели**

In [41]:
import joblib
import numpy as np
import mlflow.sklearn
import pandas as pd
from src.features import RAW_FEATURES

#1. joblib
loaded = joblib.load('models/model.joblib')
print('model.joblib загружается')

#2. MLflow Registry
model = mlflow.sklearn.load_model('models:/lab2_equipment_failure/1')
print('MLflow Registry v1 загружается')

#3. Согласованность прогнозов
df = pd.read_csv('data/raw.csv')
sample = df[RAW_FEATURES].head(5)
p1 = loaded.predict_proba(sample)
p2 = model.predict_proba(sample)
assert np.allclose(p1, p2), 'Predictions differ!'
print('Прогнозы из обоих источников совпадают')

model.joblib загружается
MLflow Registry v1 загружается
Прогнозы из обоих источников совпадают


### **Список артефактов**

In [42]:
import os

artifacts = {
    'model.joblib': 'models/model.joblib',
    'input_schema.json': 'models/input_schema.json',
    'requirements.txt': 'requirements.txt',
    'drift_report.html': 'reports/drift_report.html',
    'load_summary.csv': 'reports/load_summary.csv',
    'gradio_url.txt': 'reports/gradio_url.txt',
}

print('Артефакты проекта:')
for name, path in artifacts.items():
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f'  {name:22} ({size/1024:.1f} КБ)')
    else:
        print(f'  {name:22} НЕ НАЙДЕН')

print('\nМодули src/:')
for f in sorted(os.listdir('src')):
    if f.endswith('.py'):
        print(f'  src/{f}')

print('\nТесты tests/:')
for f in sorted(os.listdir('tests')):
    if f.endswith('.py'):
        print(f'  tests/{f}')

Артефакты проекта:
  model.joblib           (724.0 КБ)
  input_schema.json      (0.5 КБ)
  requirements.txt       (13.2 КБ)
  drift_report.html      (3052.6 КБ)
  load_summary.csv       (0.2 КБ)
  gradio_url.txt         (0.0 КБ)

Модули src/:
  src/__init__.py
  src/features.py
  src/predict.py
  src/service.py
  src/train.py

Тесты tests/:
  tests/__init__.py
  tests/test_service.py


### **Архив для сдачи**

In [43]:
import shutil
import os
from google.colab import files

archive_name = '/content/lab2_variant5'
shutil.make_archive(archive_name, 'zip', '.')
size_mb = os.path.getsize(f'{archive_name}.zip') / 1024 / 1024
print(f'Архив: {archive_name}.zip ({size_mb:.1f} МБ)')

files.download(f'{archive_name}.zip')

Архив: /content/lab2_variant5.zip (2.7 МБ)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [44]:
with open('reports/gradio_url.txt') as f:
    content = f.read().strip()
    print(f'Длина: {len(content)} символов')
    print(f'Содержимое: {content!r}')

Длина: 38 символов
Содержимое: 'https://d3cd4e87efd8f35da1.gradio.live'
